
# 03 - DEA Controlled-Substance Linkage

## Aim and scope

This notebook extends the FDA submission-event backbone by adding a **first-pass DEA controlled-substance linkage layer**.

The purpose is not to force a final thesis-ready causal dataset. The purpose is to:

- preserve and audit official DEA source materials
- document what those source materials do and do not contain
- decide the most defensible linkage unit
- construct a transparent ingredient-based linkage strategy
- export intermediate linkage outputs without overwriting the FDA backbone

## What this notebook is and is not doing

This notebook is doing:

- official-source collection and preservation in `data/raw/`
- creation of a parsed DEA reference table in a non-raw folder
- careful linkage from DEA reference data to the FDA backbone
- explicit classification of confident matches, list-I-only matches, and possible-but-uncertain candidates
- documentation of edge cases such as schedule ambiguity, preparation-specific scheduling, and incomplete DEA list coverage

This notebook is **not** doing:

- a final causal analysis
- a full rebuild of the FDA backbone
- a product-level Orange Book reconstruction
- aggressive fuzzy matching that would overstate certainty

## Why this step matters

The current FDA backbone can tell us a great deal about regulatory activity, but it cannot yet identify which records appear to involve controlled substances. A DEA linkage layer is the next natural step if the thesis is going to study whether PDUFA changed the composition of FDA-regulated drugs in ways that matter for controlled-substance exposure.


In [1]:

from pathlib import Path
from datetime import date
from urllib.request import urlopen
import re
import pandas as pd
import numpy as np
import fitz
from bs4 import BeautifulSoup
from IPython.display import display

pd.options.display.max_columns = 120
pd.options.display.float_format = lambda x: f"{x:,.4f}"


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "processed" / "fda_backbone.csv").exists() and (candidate / "code" / "notebooks").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repo root from the current working directory. "
        "Expected to find data/processed/fda_backbone.csv and code/notebooks in the repo."
    )


ROOT = find_repo_root()
FDA_BACKBONE_PATH = ROOT / "data" / "processed" / "fda_backbone.csv"
RAW_DEA_DIR = ROOT / "data" / "raw" / "dea_controlled_substances_20260315"
INTERMEDIATE_DIR = ROOT / "data" / "intermediate"
RAW_DEA_DIR.mkdir(parents=True, exist_ok=True)
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
ACCESS_DATE = date.today().isoformat()

print(f"Repo root: {ROOT}")
print(f"FDA backbone: {FDA_BACKBONE_PATH}")
print(f"DEA raw directory: {RAW_DEA_DIR}")
print(f"Intermediate directory: {INTERMEDIATE_DIR}")


Repo root: /Users/alexdelatorre/Desktop/econ580-thesis
FDA backbone: /Users/alexdelatorre/Desktop/econ580-thesis/data/processed/fda_backbone.csv
DEA raw directory: /Users/alexdelatorre/Desktop/econ580-thesis/data/raw/dea_controlled_substances_20260315
Intermediate directory: /Users/alexdelatorre/Desktop/econ580-thesis/data/intermediate



## What the FDA backbone currently affords, and what it does not

Before linking anything, it is important to restate what the FDA backbone actually contains.

This step is needed because a DEA linkage can only be as precise as the FDA-side identity fields permit. In particular, the backbone is a **submission-event panel** with **application-level aggregated ingredient strings**, not a product-level active-ingredient file.

This means the DEA linkage will have to operate through the ingredient information we currently have, rather than through a product-level NDA/ANDA/DEA registry match.


In [2]:

fda_backbone = pd.read_csv(FDA_BACKBONE_PATH, low_memory=False)

print(f"FDA backbone shape: {fda_backbone.shape[0]:,} rows x {fda_backbone.shape[1]:,} columns")

linkage_relevant_cols = [
    "ApplNo",
    "SubmissionType",
    "SubmissionNo",
    "SubmissionStatus",
    "DrugName_list",
    "ActiveIngredient_list",
    "ApplType_clean",
    "SponsorName",
]

display(fda_backbone[linkage_relevant_cols].head(8))

affordance_summary = pd.DataFrame({
    "metric": [
        "submission-event rows",
        "distinct applications",
        "rows with non-missing ActiveIngredient_list",
        "distinct ActiveIngredient_list values",
        "distinct DrugName_list values",
    ],
    "value": [
        len(fda_backbone),
        fda_backbone["ApplNo"].astype("string").str.zfill(6).nunique(dropna=True),
        int(fda_backbone["ActiveIngredient_list"].notna().sum()),
        fda_backbone["ActiveIngredient_list"].astype("string").nunique(dropna=True),
        fda_backbone["DrugName_list"].astype("string").nunique(dropna=True),
    ],
})

display(affordance_summary)


FDA backbone shape: 191,265 rows x 59 columns


,ApplNo,SubmissionType,SubmissionNo,SubmissionStatus,DrugName_list,ActiveIngredient_list,ApplType_clean,SponsorName
0,4,ORIG,1,AP,PAREDRINE,HYDROXYAMPHETAMINE HYDROBROMIDE,NDA,PHARMICS
1,4,SUPPL,10,AP,PAREDRINE,HYDROXYAMPHETAMINE HYDROBROMIDE,NDA,PHARMICS
2,4,SUPPL,11,AP,PAREDRINE,HYDROXYAMPHETAMINE HYDROBROMIDE,NDA,PHARMICS
3,159,ORIG,1,AP,SULFAPYRIDINE,SULFAPYRIDINE,NDA,LILLY
4,159,SUPPL,3,AP,SULFAPYRIDINE,SULFAPYRIDINE,NDA,LILLY
5,159,SUPPL,4,AP,SULFAPYRIDINE,SULFAPYRIDINE,NDA,LILLY
6,415,ORIG,1,AP,NaN,NaN,UNKNOWN,NaN
7,552,ORIG,1,AP,LIQUAEMIN SODIUM; LIQUAEMIN LOCK FLUSH; HEPARI...,HEPARIN SODIUM,NDA,ASPEN GLOBAL INC


,metric,value
0,submission-event rows,191265
1,distinct applications,27662
2,rows with non-missing ActiveIngredient_list,185528
3,distinct ActiveIngredient_list values,3202
4,distinct DrugName_list values,7152



### Interpretation

The FDA backbone is usable for DEA linkage, but only with care.

What it affords well:

- a stable submission-event panel
- application-level ingredient strings that can be normalized and audited
- a clean place to attach linkage flags back to the full panel

What it does **not** afford well:

- exact product-level controlled-substance classification
- formulation-specific exemptions
- historical DEA scheduling at the exact product and time level

That is why the rest of this notebook uses the FDA backbone as the master file but performs the actual linkage on **distinct ingredient aggregates and ingredient tokens**.



## DEA source selection

The notebook uses official DEA Diversion Control materials rather than informal third-party controlled-substance lists.

### Why these sources

The chosen DEA sources serve different purposes:

- the **Controlled Substance Schedules** HTML page is the scope-and-caveat source
- the **Conversion Factors for Controlled Substances** HTML page is the primary parsed reference because it is structured and includes many salt forms
- the March 2026 DEA "orange book" PDFs are secondary reference materials for schedule context and edge-case checking
- the **Table of Exempted Prescription Products** PDF is preserved because formulation-specific exemptions matter, even if the current ingredient-level linkage cannot resolve them fully

### Why this method is preferable to obvious alternatives

- It stays with official DEA material.
- It preserves the raw sources locally.
- It avoids rebuilding the FDA backbone around a different data source.
- It treats FDA Orange Book as a possible later validation layer, not a replacement for the current FDA backbone.

### Main risks and ambiguities

- DEA general-reference pages explicitly warn that their lists are **not comprehensive**.
- DEA materials may not enumerate all salts, isomers, esters, ethers, or derivatives even when those are legally controlled.
- Current DEA schedule references do not automatically recover historical schedule changes.
- Ingredient-only matching cannot fully resolve preparation-specific schedule differences or exemptions.


In [3]:

DEA_SOURCES = [
    {
        "source_name": "DEA controlled substance schedules page",
        "source_url": "https://www.deadiversion.usdoj.gov/schedules/schedules.html",
        "local_filename": "dea_controlled_substance_schedules.html",
        "source_kind": "html",
        "used_for_linkage": False,
        "used_for_caveat_audit": True,
        "scope_level": "mixed reference / schedule landing page",
        "coverage_note": "Official DEA Diversion schedules page with caveats and links; not itself a structured ingredient table.",
        "historical_note": "Reflects current reference page and links as accessed, not a historical panel.",
    },
    {
        "source_name": "DEA conversion factors for controlled substances",
        "source_url": "https://www.deadiversion.usdoj.gov/quotas/conv_factor/index.html",
        "local_filename": "dea_conversion_factors.html",
        "source_kind": "html",
        "used_for_linkage": True,
        "used_for_caveat_audit": False,
        "scope_level": "chemical / ingredient level",
        "coverage_note": "Primary parsed table for first-pass linkage because it is structured and includes many salt forms and DEA drug codes.",
        "historical_note": "Current reference page only; not a historical schedule panel.",
    },
    {
        "source_name": "DEA scheduling actions in alphabetical order",
        "source_url": "https://www.deadiversion.usdoj.gov/schedules/orangebook/a_sched_alpha.pdf",
        "local_filename": "dea_orangebook_alpha.pdf",
        "source_kind": "pdf",
        "used_for_linkage": False,
        "used_for_caveat_audit": True,
        "scope_level": "chemical / schedule action reference",
        "coverage_note": "Secondary audit source showing scheduling actions and current CSA schedule labels in alphabetical order.",
        "historical_note": "Contains scheduling action history, but this notebook does not parse it into a historical panel.",
    },
    {
        "source_name": "DEA controlled substances by drug code",
        "source_url": "https://www.deadiversion.usdoj.gov/schedules/orangebook/d_cs_drugcode.pdf",
        "local_filename": "dea_orangebook_drugcode.pdf",
        "source_kind": "pdf",
        "used_for_linkage": False,
        "used_for_caveat_audit": True,
        "scope_level": "chemical / code / schedule reference",
        "coverage_note": "Secondary audit source used to inspect preparation-specific schedule notes and edge cases such as sodium oxybate / GHB.",
        "historical_note": "Current DEA reference PDF, not a historical panel attached to the FDA approval date.",
    },
    {
        "source_name": "DEA controlled substances by CSA schedule",
        "source_url": "https://www.deadiversion.usdoj.gov/schedules/orangebook/e_cs_sched.pdf",
        "local_filename": "dea_orangebook_csa_schedule.pdf",
        "source_kind": "pdf",
        "used_for_linkage": False,
        "used_for_caveat_audit": True,
        "scope_level": "chemical / schedule reference",
        "coverage_note": "Secondary audit source for current schedule grouping.",
        "historical_note": "Current schedule grouping; not a time-varying schedule file.",
    },
    {
        "source_name": "DEA exempted prescription products table",
        "source_url": "https://www.deadiversion.usdoj.gov/schedules/exempt/exempt_rx_list.pdf",
        "local_filename": "dea_exempted_prescription_products_current.pdf",
        "source_kind": "pdf",
        "used_for_linkage": False,
        "used_for_caveat_audit": True,
        "scope_level": "product / exemption reference",
        "coverage_note": "Preserved because formulation-specific exemptions matter, but not parsed into the first-pass ingredient linkage.",
        "historical_note": "Current exempted-products table only; does not by itself recover historical exemption status for FDA records.",
    },
]


def download_if_missing(url: str, destination: Path) -> str:
    if destination.exists():
        return "already_present"
    with urlopen(url) as response:
        destination.write_bytes(response.read())
    return "downloaded"


manifest_rows = []
for spec in DEA_SOURCES:
    destination = RAW_DEA_DIR / spec["local_filename"]
    status = download_if_missing(spec["source_url"], destination)
    manifest_rows.append({
        **spec,
        "access_date": ACCESS_DATE,
        "local_path": str(destination),
        "download_status": status,
        "file_size_bytes": destination.stat().st_size if destination.exists() else pd.NA,
    })

dea_source_manifest = pd.DataFrame(manifest_rows)
display(dea_source_manifest)


,source_name,source_url,local_filename,source_kind,used_for_linkage,used_for_caveat_audit,scope_level,coverage_note,historical_note,access_date,local_path,download_status,file_size_bytes
0,DEA controlled substance schedules page,https://www.deadiversion.usdoj.gov/schedules/s...,dea_controlled_substance_schedules.html,html,False,True,mixed reference / schedule landing page,Official DEA Diversion schedules page with cav...,Reflects current reference page and links as a...,2026-03-15,/Users/alexdelatorre/Desktop/econ580-thesis/da...,already_present,53390
1,DEA conversion factors for controlled substances,https://www.deadiversion.usdoj.gov/quotas/conv...,dea_conversion_factors.html,html,True,False,chemical / ingredient level,Primary parsed table for first-pass linkage be...,Current reference page only; not a historical ...,2026-03-15,/Users/alexdelatorre/Desktop/econ580-thesis/da...,already_present,111296
2,DEA scheduling actions in alphabetical order,https://www.deadiversion.usdoj.gov/schedules/o...,dea_orangebook_alpha.pdf,pdf,False,True,chemical / schedule action reference,Secondary audit source showing scheduling acti...,"Contains scheduling action history, but this n...",2026-03-15,/Users/alexdelatorre/Desktop/econ580-thesis/da...,already_present,931948
3,DEA controlled substances by drug code,https://www.deadiversion.usdoj.gov/schedules/o...,dea_orangebook_drugcode.pdf,pdf,False,True,chemical / code / schedule reference,Secondary audit source used to inspect prepara...,"Current DEA reference PDF, not a historical pa...",2026-03-15,/Users/alexdelatorre/Desktop/econ580-thesis/da...,already_present,798674
4,DEA controlled substances by CSA schedule,https://www.deadiversion.usdoj.gov/schedules/o...,dea_orangebook_csa_schedule.pdf,pdf,False,True,chemical / schedule reference,Secondary audit source for current schedule gr...,Current schedule grouping; not a time-varying ...,2026-03-15,/Users/alexdelatorre/Desktop/econ580-thesis/da...,already_present,799406
5,DEA exempted prescription products table,https://www.deadiversion.usdoj.gov/schedules/e...,dea_exempted_prescription_products_current.pdf,pdf,False,True,product / exemption reference,Preserved because formulation-specific exempti...,Current exempted-products table only; does not...,2026-03-15,/Users/alexdelatorre/Desktop/econ580-thesis/da...,already_present,444332



### Interpretation

The raw DEA files are now preserved locally. This is important because the rest of the notebook should parse local copies rather than depend on live web state.

The source roles are intentionally split:

- one source is the **operational parsed table**
- the others are **scope, schedule, and edge-case references**

That is preferable to pretending a single DEA file is both comprehensive and perfectly parseable.



## Audit the DEA source material before parsing it into a reference table

The DEA pages themselves warn that these lists are not comprehensive. That warning is not a side note; it directly affects how strong the FDA linkage can be.

This step is needed to make the caveats explicit before any matching happens.


In [4]:

def html_text(path: Path) -> str:
    soup = BeautifulSoup(path.read_text(errors="ignore"), "lxml")
    return soup.get_text(" ", strip=True)


def excerpt_around(text: str, phrase: str, window: int = 700) -> str:
    lower_text = text.lower()
    lower_phrase = phrase.lower()
    idx = lower_text.find(lower_phrase)
    if idx == -1:
        return "Phrase not found."
    start = max(0, idx - window // 2)
    end = min(len(text), idx + len(phrase) + window // 2)
    return text[start:end]


def pdf_page_count(path: Path) -> int:
    return fitz.open(path).page_count


schedule_text = html_text(RAW_DEA_DIR / "dea_controlled_substance_schedules.html")
schedule_caveat_excerpt = excerpt_around(schedule_text, "not a comprehensive list", window=900)

pdf_audit = []
for filename in dea_source_manifest.loc[dea_source_manifest["source_kind"].eq("pdf"), "local_filename"]:
    path = RAW_DEA_DIR / filename
    pdf_audit.append({
        "local_filename": filename,
        "page_count": pdf_page_count(path),
        "file_size_bytes": path.stat().st_size,
    })
pdf_audit = pd.DataFrame(pdf_audit)

print("DEA schedules-page caveat excerpt")
print(schedule_caveat_excerpt)
print()
display(pdf_audit)


DEA schedules-page caveat excerpt
sted in Schedule IV and consist primarily of preparations containing limited quantities of certain narcotics. Examples of Schedule V substances include: cough preparations containing not more than 200 milligrams of codeine per 100 milliliters or per 100 grams (Robitussin AC®, Phenergan with Codeine®), and ezogabine. Lists of Scheduling Actions, Controlled Substances, Regulated Chemicals (PDF) (March 2026) This document is a general reference and not a comprehensive list. This list describes the basic or parent chemical and does not describe the salts, isomers and salts of isomers, esters, ethers and derivatives which may also be controlled substances. Scheduling Actions Controlled Substances List I and II Regulated Chemicals Alphabetical Order Alphabetical Order Alphabetical Order Chronological Order Controlled Substance Code Number DEA Chemical Code Number CSA Schedule List Number Illicit Uses and Threshold Qu



,local_filename,page_count,file_size_bytes
0,dea_orangebook_alpha.pdf,23,931948
1,dea_orangebook_drugcode.pdf,22,798674
2,dea_orangebook_csa_schedule.pdf,22,799406
3,dea_exempted_prescription_products_current.pdf,13,444332



### Interpretation

The schedules page explicitly warns that the DEA lists are **not comprehensive** and may not enumerate salts, isomers, esters, ethers, and derivatives even when those are controlled.

That has two direct implications for the linkage:

1. a failure to match a DEA name does **not** prove that an FDA ingredient is non-controlled
2. certain positive matches will still need uncertainty notes when schedule assignment depends on preparation details or omitted isomer naming

This is why the notebook will separate:

- **confident ingredient-level matches**
- **List I chemical matches that are not the same thing as scheduled controlled substances**
- **possible-only substring candidates for manual review**



## Parse the DEA reference material

The primary parsed source in this first pass is the DEA **conversion factors** HTML table.

### Why this method is preferable

- It is structured enough to parse reproducibly with `pandas.read_html`.
- It includes many salt forms and DEA drug codes.
- It is much easier to audit than an opaque hand-built custom list.

### Why this is still incomplete

- It includes `List I` chemicals alongside scheduled substances.
- It does not, by itself, resolve preparation-specific schedule differences.
- It is still a current reference, not a historical schedule file.

So the parsed DEA reference below is a **first-pass operational linkage table**, not a legally exhaustive ontology of controlled substances.


In [5]:

SCHEDULED_CSA_VALUES = {"I", "II", "III", "IV", "V"}
SALT_TOKENS = {
    "ACETATE", "ALKALOID", "ANHYDROUS", "ASPARTATE", "BASE", "BENZOATE", "BESYLATE", "BITARTRATE",
    "BROMIDE", "CALCIUM", "CAMSYLATE", "CHLORIDE", "CITRATE", "DECANOATE", "DIHYDRATE", "DIMESYLATE",
    "DINITRATE", "EDISYLATE", "FUMARATE", "GLUCEPTATE", "HEMISULFATE", "HEMISULPHATE", "HBR", "HCL",
    "HYDRATE", "HYDROBROMIDE", "HYDROCHLORIDE", "LACTATE", "MALEATE", "MAGNESIUM", "MESYLATE",
    "MONOHYDRATE", "NAPSYLATE", "NITRATE", "PAMOATE", "PHOSPHATE", "POTASSIUM", "RESINATE", "SACCHARATE",
    "SEBACATE", "SODIUM", "SUCCINATE", "SULFATE", "SULPHATE", "TARTRATE", "TEREPHTHALATE", "TOSYLATE",
    "TRIHYDRATE", "VALERATE", "XINAFOATE",
}
FORMULA_RE = re.compile(r"^(\d+|H\d+O)$")


def clean_string_series(s: pd.Series) -> pd.Series:
    return (
        s.astype("string")
        .str.replace("﻿", "", regex=False)
        .str.strip()
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    )


def normalize_text(text):
    if pd.isna(text):
        return pd.NA
    text = str(text).upper().replace("&", " AND ")
    text = re.sub(r"\([^)]*\)", " ", text)
    text = re.sub(r"[^A-Z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text if text else pd.NA


def split_alias_candidates(text):
    if pd.isna(text):
        return []
    raw = str(text)
    pieces = []
    pieces.extend(re.findall(r"\(([^)]*)\)", raw))
    if "," in raw:
        pieces.extend(raw.split(","))

    aliases = []
    for fragment in pieces:
        for part in re.split(r";|and|/", fragment, flags=re.IGNORECASE):
            norm = normalize_text(part)
            if pd.notna(norm) and len(norm) >= 4:
                aliases.append(norm)
    return list(dict.fromkeys(aliases))


def root_normalize(norm_text):
    if pd.isna(norm_text):
        return pd.NA
    tokens = str(norm_text).split()
    while tokens and (tokens[-1] in SALT_TOKENS or FORMULA_RE.fullmatch(tokens[-1])):
        tokens.pop()
    root = " ".join(tokens).strip()
    return root if root else norm_text


conversion_tables = pd.read_html(RAW_DEA_DIR / "dea_conversion_factors.html")
dea_conversion_definitions = conversion_tables[0].copy()
dea_conversion_raw = conversion_tables[1].copy()

for col in ["Controlled Substance", "Schedule", "Drug Code"]:
    dea_conversion_raw[col] = clean_string_series(dea_conversion_raw[col])
dea_conversion_raw["Conversion Factor"] = pd.to_numeric(dea_conversion_raw["Conversion Factor"], errors="coerce")

dea_reference_rows = []
for _, row in dea_conversion_raw.iterrows():
    core_name_norm = normalize_text(row["Controlled Substance"])
    dea_reference_rows.append({
        "dea_source_name_raw": row["Controlled Substance"],
        "dea_candidate_name_norm": core_name_norm,
        "dea_candidate_type": "core_name",
        "dea_root_norm": root_normalize(core_name_norm),
        "dea_current_schedule": row["Schedule"],
        "dea_drug_code": row["Drug Code"],
        "dea_conversion_factor": row["Conversion Factor"],
        "dea_is_csa_scheduled": row["Schedule"] in SCHEDULED_CSA_VALUES,
        "dea_is_list_i_chemical": row["Schedule"] == "List I",
        "dea_source_file": "dea_conversion_factors.html",
        "dea_source_url": dea_source_manifest.loc[
            dea_source_manifest["local_filename"].eq("dea_conversion_factors.html"), "source_url"
        ].iloc[0],
    })
    for alias in split_alias_candidates(row["Controlled Substance"]):
        dea_reference_rows.append({
            "dea_source_name_raw": row["Controlled Substance"],
            "dea_candidate_name_norm": alias,
            "dea_candidate_type": "alias_fragment",
            "dea_root_norm": root_normalize(alias),
            "dea_current_schedule": row["Schedule"],
            "dea_drug_code": row["Drug Code"],
            "dea_conversion_factor": row["Conversion Factor"],
            "dea_is_csa_scheduled": row["Schedule"] in SCHEDULED_CSA_VALUES,
            "dea_is_list_i_chemical": row["Schedule"] == "List I",
            "dea_source_file": "dea_conversion_factors.html",
            "dea_source_url": dea_source_manifest.loc[
                dea_source_manifest["local_filename"].eq("dea_conversion_factors.html"), "source_url"
            ].iloc[0],
        })

dea_reference = pd.DataFrame(dea_reference_rows).drop_duplicates().reset_index(drop=True)

display(dea_conversion_raw.head(10))
display(dea_reference.head(12))


,Controlled Substance,Schedule,Drug Code,Conversion Factor
0,(+/-)cis-4-Methylaminorex,I,1590,1.0000
1,(+/-)cis-4-Methylaminorex Hydrochloride,I,1590,0.8300
2,"1-(1,3-benzodioxol-5-yl)-2-(ethylamino)propan-...",I,7547,1.0000
3,"1-(1,3-benzodioxol-5-yl)-2-(ethylamino)propan-...",I,7547,0.8600
4,"(1-(4-fluorobenzyl)-1H-indol-3-yl)(2,2,3,3-tet...",I,7014,1.0000
5,1-(4-cyanobutyl)-N-(2-phenylpropan-2-yl)-1H-in...,I,7089,1.0000
6,1-(5-fluoropentyl)-1H-indazol-3-yl](naphthalen...,I,7024,1.0000
7,1-(5-fluoropentyl)-3-(1-naphthoyl)indole (AM2201),I,7201,1.0000
8,1-(5-fluoropentyl)-3-(2-iodobenzoyl)indole (AM...,I,7694,1.0000
9,1-(5-fluoropentyl)-N-(2-phenylpropan-2-yl)-1H-...,I,7083,1.0000


,dea_source_name_raw,dea_candidate_name_norm,dea_candidate_type,dea_root_norm,dea_current_schedule,dea_drug_code,dea_conversion_factor,dea_is_csa_scheduled,dea_is_list_i_chemical,dea_source_file,dea_source_url
0,(+/-)cis-4-Methylaminorex,CIS 4 METHYLAMINOREX,core_name,CIS 4 METHYLAMINOREX,I,1590,1.0000,True,False,dea_conversion_factors.html,https://www.deadiversion.usdoj.gov/quotas/conv...
1,(+/-)cis-4-Methylaminorex Hydrochloride,CIS 4 METHYLAMINOREX HYDROCHLORIDE,core_name,CIS 4 METHYLAMINOREX,I,1590,0.8300,True,False,dea_conversion_factors.html,https://www.deadiversion.usdoj.gov/quotas/conv...
2,"1-(1,3-benzodioxol-5-yl)-2-(ethylamino)propan-...",1 2 PROPAN 1 ONE,core_name,1 2 PROPAN 1 ONE,I,7547,1.0000,True,False,dea_conversion_factors.html,https://www.deadiversion.usdoj.gov/quotas/conv...
3,"1-(1,3-benzodioxol-5-yl)-2-(ethylamino)propan-...",1 3 BENZODIOXOL 5 YL,alias_fragment,1 3 BENZODIOXOL 5 YL,I,7547,1.0000,True,False,dea_conversion_factors.html,https://www.deadiversion.usdoj.gov/quotas/conv...
4,"1-(1,3-benzodioxol-5-yl)-2-(ethylamino)propan-...",ETHYLAMINO,alias_fragment,ETHYLAMINO,I,7547,1.0000,True,False,dea_conversion_factors.html,https://www.deadiversion.usdoj.gov/quotas/conv...
5,"1-(1,3-benzodioxol-5-yl)-2-(ethylamino)propan-...",ETHYLONE,alias_fragment,ETHYLONE,I,7547,1.0000,True,False,dea_conversion_factors.html,https://www.deadiversion.usdoj.gov/quotas/conv...
6,"1-(1,3-benzodioxol-5-yl)-2-(ethylamino)propan-...",3 BENZODIOXOL 5 YL 2 PROPAN 1 ONE,alias_fragment,3 BENZODIOXOL 5 YL 2 PROPAN 1 ONE,I,7547,1.0000,True,False,dea_conversion_factors.html,https://www.deadiversion.usdoj.gov/quotas/conv...
7,"1-(1,3-benzodioxol-5-yl)-2-(ethylamino)propan-...",1 2 PROPAN 1 ONE HYDROCHLORIDE,core_name,1 2 PROPAN 1 ONE,I,7547,0.8600,True,False,dea_conversion_factors.html,https://www.deadiversion.usdoj.gov/quotas/conv...
8,"1-(1,3-benzodioxol-5-yl)-2-(ethylamino)propan-...",1 3 BENZODIOXOL 5 YL,alias_fragment,1 3 BENZODIOXOL 5 YL,I,7547,0.8600,True,False,dea_conversion_factors.html,https://www.deadiversion.usdoj.gov/quotas/conv...
9,"1-(1,3-benzodioxol-5-yl)-2-(ethylamino)propan-...",ETHYLAMINO,alias_fragment,ETHYLAMINO,I,7547,0.8600,True,False,dea_conversion_factors.html,https://www.deadiversion.usdoj.gov/quotas/conv...



### Interpretation

The parsed DEA reference now separates:

- the **raw DEA source name**
- normalized candidate strings used for matching
- root-normalized strings used for conservative salt/hydrate normalization
- schedule flags that distinguish **scheduled controlled substances** from **List I chemicals**

This distinction is essential. A DEA `List I` chemical match is still a meaningful regulatory signal, but it is not the same thing as a scheduled controlled-substance match.



## Audit the DEA reference content and inspect edge cases from secondary DEA materials

This step is needed for two reasons:

1. to understand what the parsed reference actually contains
2. to identify cases where the secondary DEA PDFs reveal schedule nuances that the primary parsed table does not fully resolve

The edge-case audit is especially important because the thesis question cares about controlled-substance involvement, and ingredient-level schedule assignment can be wrong if preparation-specific scheduling is ignored.


In [6]:

def pdf_text(path: Path) -> str:
    doc = fitz.open(path)
    return "".join(page.get_text() for page in doc)


dea_reference_summary = pd.DataFrame({
    "metric": [
        "raw rows in DEA conversion-factor table",
        "candidate rows in parsed DEA reference",
        "unique DEA source names",
        "unique DEA candidate strings",
        "candidate rows flagged as CSA schedules I-V",
        "candidate rows flagged as List I chemicals",
    ],
    "value": [
        len(dea_conversion_raw),
        len(dea_reference),
        dea_reference["dea_source_name_raw"].nunique(),
        dea_reference["dea_candidate_name_norm"].nunique(),
        int(dea_reference["dea_is_csa_scheduled"].sum()),
        int(dea_reference["dea_is_list_i_chemical"].sum()),
    ],
})

schedule_counts = (
    dea_reference[["dea_source_name_raw", "dea_current_schedule"]]
    .drop_duplicates()
    ["dea_current_schedule"]
    .value_counts(dropna=False)
    .rename_axis("dea_current_schedule")
    .reset_index(name="unique_dea_source_names")
)

drugcode_text = pdf_text(RAW_DEA_DIR / "dea_orangebook_drugcode.pdf")
edge_case_terms = {
    "Gamma Hydroxybutyric Acid / sodium oxybate": "GAMMA HYDROXYBUTYRIC ACID",
    "Codeine preparations": "CODEINE PREPARATIONS",
    "Dihydrocodeine preparations": "DIHYDROCODEINE PREPARATIONS",
    "Opium preparations": "OPIUM PREPARATIONS",
}

edge_case_rows = []
for label, needle in edge_case_terms.items():
    upper_text = drugcode_text.upper()
    idx = upper_text.find(needle)
    excerpt = drugcode_text[idx:idx + 700] if idx != -1 else "Needle not found in PDF text."
    edge_case_rows.append({
        "edge_case_label": label,
        "needle": needle,
        "found_in_drugcode_pdf": idx != -1,
        "excerpt": excerpt,
    })
edge_case_evidence = pd.DataFrame(edge_case_rows)

display(dea_reference_summary)
display(schedule_counts)
display(edge_case_evidence)


,metric,value
0,raw rows in DEA conversion-factor table,741
1,candidate rows in parsed DEA reference,1210
2,unique DEA source names,741
3,unique DEA candidate strings,919
4,candidate rows flagged as CSA schedules I-V,1197
5,candidate rows flagged as List I chemicals,13


,dea_current_schedule,unique_dea_source_names
0,I,399
1,II,182
2,IV,110
3,III,32
4,List I,11
5,V,7


,edge_case_label,needle,found_in_drugcode_pdf,excerpt
0,Gamma Hydroxybutyric Acid / sodium oxybate,GAMMA HYDROXYBUTYRIC ACID,True,Gamma Hydroxybutyric Acid \n2010 \nI \nN \nGHB...
1,Codeine preparations,CODEINE PREPARATIONS,True,Codeine preparations - 200 mg/(100 ml or 100 g...
2,Dihydrocodeine preparations,DIHYDROCODEINE PREPARATIONS,True,Dihydrocodeine preparations 100mg/(100 ml or 1...
3,Opium preparations,OPIUM PREPARATIONS,True,Opium preparations - 100 mg/(100 ml or 100 gm)...



### Interpretation

A few important issues show up immediately.

First, the parsed DEA reference includes `List I` chemicals, so the notebook cannot use a simple "any DEA match = controlled substance" rule.

Second, the secondary DEA drug-code PDF shows preparation-specific schedule distinctions. The sodium oxybate / GHB case is especially important: ingredient-level matching can identify a strong DEA-controlled-substance signal, but the **schedule attached to the ingredient may depend on preparation context**. The same problem can arise for codeine-related preparations.

That means the linkage should prioritize **controlled-substance involvement** over false precision about the final schedule in every matched FDA row.



## Decide the linkage unit

### Goal

Choose the most defensible unit for a first-pass FDA-to-DEA bridge.

### Choice

The linkage will use **active ingredient tokens** as the primary bridge.

### Why this is preferable to obvious alternatives

- **Submission-event level** is too downstream. The DEA reference is not keyed to FDA submission IDs.
- **Application level** is too coarse if different products or formulations share an application.
- **Product level** would be ideal in some settings, but the current backbone only carries application-level aggregated ingredient strings, not a clean product-level ingredient table.
- **Active ingredient** is the closest common semantic object between the current FDA backbone and the DEA reference materials.

### Operational implementation

Operationally, the notebook will:

1. work on distinct `ActiveIngredient_list` strings from the FDA backbone
2. split those strings into ingredient tokens conservatively
3. match tokens to DEA reference candidates
4. aggregate the result back to the distinct ingredient-string level
5. merge that back to the full FDA submission-event panel

### Main risk

This means the DEA linkage is only as precise as the `ActiveIngredient_list` field. Because that field is already an application-level aggregate in the FDA backbone, the linked result is best understood as an **application-ingredient signal propagated to submission-event rows**, not a perfect product-level classification.


In [7]:

fda_backbone_work = fda_backbone.copy()
for col in ["ApplNo", "SubmissionNo", "SubmissionType", "SubmissionStatus", "DrugName_list", "ActiveIngredient_list"]:
    if col in fda_backbone_work.columns:
        fda_backbone_work[col] = clean_string_series(fda_backbone_work[col])

active_ingredient_examples = fda_backbone_work["ActiveIngredient_list"].dropna().drop_duplicates()
unit_profile = pd.DataFrame({
    "metric": [
        "distinct ActiveIngredient_list values",
        "distinct ActiveIngredient_list values containing ';'",
        "distinct ActiveIngredient_list values containing '/'",
        "distinct ActiveIngredient_list values containing ','",
        "distinct ActiveIngredient_list values containing ' AND '",
    ],
    "value": [
        len(active_ingredient_examples),
        int(active_ingredient_examples.str.contains(";", regex=False).sum()),
        int(active_ingredient_examples.str.contains("/", regex=False).sum()),
        int(active_ingredient_examples.str.contains(",", regex=False).sum()),
        int(active_ingredient_examples.str.contains(" AND ", regex=False).sum()),
    ],
})

semicolon_examples = active_ingredient_examples[active_ingredient_examples.str.contains(";", regex=False)].head(15)

display(unit_profile)
display(semicolon_examples.to_frame(name="ActiveIngredient_list_example"))


,metric,value
0,distinct ActiveIngredient_list values,3202
1,distinct ActiveIngredient_list values containi...,686
2,distinct ActiveIngredient_list values containi...,7
3,distinct ActiveIngredient_list values containi...,74
4,distinct ActiveIngredient_list values containi...,18


,ActiveIngredient_list_example
175,SULFADIAZINE; SULFADIAZINE SODIUM
225,ALCOHOL; DEXTROSE
379,HOMATROPINE METHYLBROMIDE; HYDROCODONE BITARTRATE
534,TRIPLE SULFA (SULFABENZAMIDE;SULFACETAMIDE;SUL...
597,TRIPELENNAMINE HYDROCHLORIDE; TRIPELENNAMINE C...
659,CHLOROQUINE PHOSPHATE; CHLOROQUINE HYDROCHLORIDE
720,FERROUS SULFATE; FOLIC ACID
781,ASCORBIC ACID; BIOTIN; CYANOCOBALAMIN; DEXPANT...
1011,LIDOCAINE HYDROCHLORIDE; EPINEPHRINE; LIDOCAIN...
1141,CAFFEINE; ERGOTAMINE TARTRATE



### Interpretation

The FDA ingredient field is clearly rich enough to support a first-pass bridge, but it is also clearly messy enough that tokenization choices matter.

The notebook therefore uses a conservative tokenization rule: **split on semicolons first**, keep the raw token text, and avoid aggressive decomposition of commas or slashes that could destroy meaning.

That choice is preferable here because it preserves auditability. Some multi-ingredient cases will remain imperfect, but the notebook will show them rather than hide them.



## Build normalized matching fields from the FDA backbone

This step creates a distinct ingredient-string table and a token-level audit table from the FDA side.

The notebook preserves raw ingredient strings and token strings while also building normalized fields for exact and root-normalized matching.


In [8]:

def split_fda_ingredients(text):
    if pd.isna(text):
        return []
    return [part.strip() for part in str(text).split(";") if part.strip()]


fda_ingredient_aggregates = pd.DataFrame({
    "ActiveIngredient_list": fda_backbone_work["ActiveIngredient_list"].dropna().drop_duplicates().sort_values()
}).reset_index(drop=True)

fda_token_rows = []
for ingredient_string in fda_ingredient_aggregates["ActiveIngredient_list"]:
    for token_order, token_raw in enumerate(split_fda_ingredients(ingredient_string), start=1):
        token_norm = normalize_text(token_raw)
        fda_token_rows.append({
            "ActiveIngredient_list": ingredient_string,
            "ingredient_token_order": token_order,
            "fda_ingredient_token_raw": token_raw,
            "fda_token_norm": token_norm,
            "fda_token_root_norm": root_normalize(token_norm),
        })

fda_ingredient_tokens = pd.DataFrame(fda_token_rows)

display(fda_ingredient_aggregates.head(10))
display(fda_ingredient_tokens.head(15))


,ActiveIngredient_list
0,ABACAVIR SULFATE
1,ABACAVIR SULFATE; DOLUTEGRAVIR SODIUM; LAMIVUDINE
2,ABACAVIR SULFATE; LAMIVUDINE
3,ABACAVIR SULFATE; LAMIVUDINE; ZIDOVUDINE
4,ABACAVIR || DOLUTEGRAVIR || LAMIVUDINE
5,"ABACAVIR, DOLUTEGRAVIR, LAMIVUDINE"
6,ABACAVIR; DOLUTEGRAVIR; LAMIVUDINE
7,ABACAVIR; LAMIVUDINE
8,ABACAVIR;DOLUTEGRAVIR;LAMIVUDINE
9,ABACAVIR;LAMIVUDINE


,ActiveIngredient_list,ingredient_token_order,fda_ingredient_token_raw,fda_token_norm,fda_token_root_norm
0,ABACAVIR SULFATE,1,ABACAVIR SULFATE,ABACAVIR SULFATE,ABACAVIR
1,ABACAVIR SULFATE; DOLUTEGRAVIR SODIUM; LAMIVUDINE,1,ABACAVIR SULFATE,ABACAVIR SULFATE,ABACAVIR
2,ABACAVIR SULFATE; DOLUTEGRAVIR SODIUM; LAMIVUDINE,2,DOLUTEGRAVIR SODIUM,DOLUTEGRAVIR SODIUM,DOLUTEGRAVIR
3,ABACAVIR SULFATE; DOLUTEGRAVIR SODIUM; LAMIVUDINE,3,LAMIVUDINE,LAMIVUDINE,LAMIVUDINE
4,ABACAVIR SULFATE; LAMIVUDINE,1,ABACAVIR SULFATE,ABACAVIR SULFATE,ABACAVIR
5,ABACAVIR SULFATE; LAMIVUDINE,2,LAMIVUDINE,LAMIVUDINE,LAMIVUDINE
6,ABACAVIR SULFATE; LAMIVUDINE; ZIDOVUDINE,1,ABACAVIR SULFATE,ABACAVIR SULFATE,ABACAVIR
7,ABACAVIR SULFATE; LAMIVUDINE; ZIDOVUDINE,2,LAMIVUDINE,LAMIVUDINE,LAMIVUDINE
8,ABACAVIR SULFATE; LAMIVUDINE; ZIDOVUDINE,3,ZIDOVUDINE,ZIDOVUDINE,ZIDOVUDINE
9,ABACAVIR || DOLUTEGRAVIR || LAMIVUDINE,1,ABACAVIR || DOLUTEGRAVIR || LAMIVUDINE,ABACAVIR DOLUTEGRAVIR LAMIVUDINE,ABACAVIR DOLUTEGRAVIR LAMIVUDINE



### Interpretation

At this point the FDA side is ready for linkage. The key thing to note is that the notebook keeps both:

- the raw ingredient token for manual audit
- the normalized token and root-normalized token for matching

This is preferable to replacing the raw field with a cleaned field. It keeps the linkage auditable, which matters for any future manual review of controlled-substance cases.



## Perform confident linkage stages: exact and normalized-root exact

These are the only stages that will count as **first-pass confident matches**.

### Why this method is preferable

- exact matches are the cleanest signal
- root-normalized exact matches help recover salt/hydrate variants without jumping straight to fuzzy logic
- both stages remain easy to audit because they preserve the matched DEA names and FDA tokens

### Main risks

- even exact ingredient matches may still have schedule ambiguity when DEA materials distinguish preparations from the parent chemical
- root normalization can blur chemically meaningful formulation differences, so it must remain conservative and documented


In [9]:

TOKEN_KEY = [
    "ActiveIngredient_list",
    "ingredient_token_order",
    "fda_ingredient_token_raw",
    "fda_token_norm",
    "fda_token_root_norm",
]


def collapse_unique(values):
    values = [str(v) for v in pd.Series(values).dropna().astype(str).unique() if str(v).strip()]
    return " | ".join(values) if values else pd.NA


def summarize_match_stage(candidate_df: pd.DataFrame, stage_label: str) -> pd.DataFrame:
    if candidate_df.empty:
        return pd.DataFrame(columns=TOKEN_KEY)

    grouped = candidate_df.groupby(TOKEN_KEY, dropna=False)
    out = grouped.apply(
        lambda g: pd.Series({
            "dea_scheduled_substance_names": collapse_unique(g.loc[g["dea_is_csa_scheduled"], "dea_source_name_raw"]),
            "dea_scheduled_candidate_names": collapse_unique(g.loc[g["dea_is_csa_scheduled"], "dea_candidate_name_norm"]),
            "dea_scheduled_current_schedules": collapse_unique(g.loc[g["dea_is_csa_scheduled"], "dea_current_schedule"]),
            "dea_scheduled_drug_codes": collapse_unique(g.loc[g["dea_is_csa_scheduled"], "dea_drug_code"]),
            "dea_list_i_substance_names": collapse_unique(g.loc[g["dea_is_list_i_chemical"], "dea_source_name_raw"]),
            "dea_list_i_candidate_names": collapse_unique(g.loc[g["dea_is_list_i_chemical"], "dea_candidate_name_norm"]),
            "dea_list_i_drug_codes": collapse_unique(g.loc[g["dea_is_list_i_chemical"], "dea_drug_code"]),
            "dea_source_name_count": g["dea_source_name_raw"].nunique(),
            "dea_scheduled_candidate_count": int(g["dea_is_csa_scheduled"].sum()),
            "dea_list_i_candidate_count": int(g["dea_is_list_i_chemical"].sum()),
            "dea_candidate_type_summary": collapse_unique(g["dea_candidate_type"]),
        })
    ).reset_index()

    out["dea_stage_label"] = stage_label
    out["dea_confident_match_flag"] = out["dea_scheduled_candidate_count"] > 0
    out["dea_list_i_only_flag"] = (out["dea_scheduled_candidate_count"] == 0) & (out["dea_list_i_candidate_count"] > 0)

    if stage_label == "exact":
        out["dea_match_method"] = np.where(
            out["dea_candidate_type_summary"].fillna("").str.contains("alias_fragment", regex=False),
            "exact_alias_fragment",
            "exact",
        )
    else:
        out["dea_match_method"] = "normalized_root_exact"

    return out


exact_candidates = fda_ingredient_tokens.merge(
    dea_reference,
    left_on="fda_token_norm",
    right_on="dea_candidate_name_norm",
    how="left",
)
exact_candidates = exact_candidates[exact_candidates["dea_source_name_raw"].notna()].copy()
exact_token_summary = summarize_match_stage(exact_candidates, stage_label="exact")

matched_exact_keys = exact_token_summary[TOKEN_KEY].drop_duplicates() if len(exact_token_summary) else pd.DataFrame(columns=TOKEN_KEY)
remaining_after_exact = fda_ingredient_tokens.merge(
    matched_exact_keys.assign(_matched=1),
    on=TOKEN_KEY,
    how="left",
)
remaining_after_exact = remaining_after_exact[remaining_after_exact["_matched"].isna()].drop(columns="_matched")

root_candidates = remaining_after_exact.merge(
    dea_reference,
    left_on="fda_token_root_norm",
    right_on="dea_root_norm",
    how="left",
)
root_candidates = root_candidates[root_candidates["dea_source_name_raw"].notna()].copy()
root_token_summary = summarize_match_stage(root_candidates, stage_label="root")

confident_token_matches = pd.concat([exact_token_summary, root_token_summary], ignore_index=True)
confident_token_matches = confident_token_matches.sort_values(TOKEN_KEY).drop_duplicates(TOKEN_KEY)

confident_method_counts = (
    confident_token_matches["dea_match_method"]
    .value_counts(dropna=False)
    .rename_axis("dea_match_method")
    .reset_index(name="token_count")
)

display(confident_method_counts)
display(confident_token_matches.head(20))


,dea_match_method,token_count
0,exact,176
1,normalized_root_exact,5
2,exact_alias_fragment,2


,ActiveIngredient_list,ingredient_token_order,fda_ingredient_token_raw,fda_token_norm,fda_token_root_norm,dea_scheduled_substance_names,dea_scheduled_candidate_names,dea_scheduled_current_schedules,dea_scheduled_drug_codes,dea_list_i_substance_names,dea_list_i_candidate_names,dea_list_i_drug_codes,dea_source_name_count,dea_scheduled_candidate_count,dea_list_i_candidate_count,dea_candidate_type_summary,dea_stage_label,dea_confident_match_flag,dea_list_i_only_flag,dea_match_method
0,ACETAMINOPHEN; ASPIRIN; CODEINE PHOSPHATE,3,CODEINE PHOSPHATE,CODEINE PHOSPHATE,CODEINE,Codeine Phosphate (1 1/2 H2O) | Codeine Phosph...,CODEINE PHOSPHATE,II,9050,<NA>,<NA>,<NA>,2,2,0,core_name,exact,True,False,exact
1,ACETAMINOPHEN; BUTALBITAL; CAFFEINE; CODEINE P...,4,CODEINE PHOSPHATE,CODEINE PHOSPHATE,CODEINE,Codeine Phosphate (1 1/2 H2O) | Codeine Phosph...,CODEINE PHOSPHATE,II,9050,<NA>,<NA>,<NA>,2,2,0,core_name,exact,True,False,exact
2,ACETAMINOPHEN; BUTALBITAL; CAFFEINE; CODEINE P...,4,CODEINE PHOSPHATE,CODEINE PHOSPHATE,CODEINE,Codeine Phosphate (1 1/2 H2O) | Codeine Phosph...,CODEINE PHOSPHATE,II,9050,<NA>,<NA>,<NA>,2,2,0,core_name,exact,True,False,exact
3,ACETAMINOPHEN; BUTALBITAL; CAFFEINE; CODEINE P...,8,CODEINE PHOSPHATE,CODEINE PHOSPHATE,CODEINE,Codeine Phosphate (1 1/2 H2O) | Codeine Phosph...,CODEINE PHOSPHATE,II,9050,<NA>,<NA>,<NA>,2,2,0,core_name,exact,True,False,exact
4,ACETAMINOPHEN; CAFFEINE; DIHYDROCODEINE BITART...,3,DIHYDROCODEINE BITARTRATE,DIHYDROCODEINE BITARTRATE,DIHYDROCODEINE,Dihydrocodeine Bitartrate | Dihydrocodeine Bit...,DIHYDROCODEINE BITARTRATE,II,9120,<NA>,<NA>,<NA>,2,2,0,core_name,exact,True,False,exact
5,ACETAMINOPHEN; CLEMASTINE FUMARATE; PSEUDOEPHE...,3,PSEUDOEPHEDRINE HYDROCHLORIDE,PSEUDOEPHEDRINE HYDROCHLORIDE,PSEUDOEPHEDRINE,<NA>,<NA>,<NA>,<NA>,Pseudoephedrine Hydrochloride,PSEUDOEPHEDRINE HYDROCHLORIDE,8112,1,0,1,core_name,exact,False,True,exact
6,ACETAMINOPHEN; CODEINE PHOSPHATE,2,CODEINE PHOSPHATE,CODEINE PHOSPHATE,CODEINE,Codeine Phosphate (1 1/2 H2O) | Codeine Phosph...,CODEINE PHOSPHATE,II,9050,<NA>,<NA>,<NA>,2,2,0,core_name,exact,True,False,exact
7,ACETAMINOPHEN; DEXBROMPHENIRAMINE MALEATE; PSE...,3,PSEUDOEPHEDRINE SULFATE,PSEUDOEPHEDRINE SULFATE,PSEUDOEPHEDRINE,<NA>,<NA>,<NA>,<NA>,Pseudoephedrine Sulfate (Dibasic),PSEUDOEPHEDRINE SULFATE,8112,1,0,1,core_name,exact,False,True,exact
8,ACETAMINOPHEN; HYDROCODONE BITARTRATE,2,HYDROCODONE BITARTRATE,HYDROCODONE BITARTRATE,HYDROCODONE,Hydrocodone Bitartrate (2 1/2 H2O),HYDROCODONE BITARTRATE,II,9193,<NA>,<NA>,<NA>,1,1,0,core_name,exact,True,False,exact
9,ACETAMINOPHEN; OXYCODONE HYDROCHLORIDE,2,OXYCODONE HYDROCHLORIDE,OXYCODONE HYDROCHLORIDE,OXYCODONE,Oxycodone Hydrochloride,OXYCODONE HYDROCHLORIDE,II,9143,<NA>,<NA>,<NA>,1,1,0,core_name,exact,True,False,exact



### Interpretation

These confident matches are the core of the first-pass linkage. They identify FDA ingredient tokens that can be tied back to DEA reference material without relying on loose heuristics.

Even here, confidence should be interpreted carefully:

- a confident **ingredient** match is not always a confident **final schedule assignment**
- some matches may still need edge-case notes because DEA schedule treatment can vary by preparation or historical period

That is why the notebook still audits substring candidates separately instead of forcing them into the same certainty bucket.



## Build conservative substring candidates for manual review only

The DEA schedules page explicitly warns that not all salts, isomers, esters, ethers, and derivatives are listed.

That means some real controlled-substance-related FDA ingredients may be missed by exact or root-exact matching alone. At the same time, loose substring logic can create obvious false positives.

So this stage is intentionally conservative and **audit-only**:

- it does **not** count as a confident match
- it is meant to surface likely parent/isomer/derivative candidates for later review
- its main value is diagnostic, not classificatory


In [10]:

matched_confident_keys = confident_token_matches[TOKEN_KEY].drop_duplicates() if len(confident_token_matches) else pd.DataFrame(columns=TOKEN_KEY)
remaining_after_confident = fda_ingredient_tokens.merge(
    matched_confident_keys.assign(_matched=1),
    on=TOKEN_KEY,
    how="left",
)
remaining_after_confident = remaining_after_confident[remaining_after_confident["_matched"].isna()].drop(columns="_matched")

substring_candidate_rows = []
for _, row in remaining_after_confident.iterrows():
    fda_norm = row["fda_token_norm"]
    fda_root = row["fda_token_root_norm"]
    if pd.isna(fda_norm):
        continue

    candidate_pool = dea_reference.loc[dea_reference["dea_is_csa_scheduled"]].copy()
    candidate_pool = candidate_pool[candidate_pool["dea_candidate_name_norm"].str.len() >= 8]

    candidate_pool = candidate_pool.loc[
        candidate_pool["dea_candidate_name_norm"].map(lambda x: isinstance(x, str) and x in fda_norm)
        | candidate_pool["dea_root_norm"].map(lambda x: isinstance(x, str) and x in fda_root if pd.notna(fda_root) else False)
    ].copy()

    if candidate_pool.empty:
        continue

    substring_candidate_rows.append({
        **row.to_dict(),
        "dea_possible_candidate_substance_names": collapse_unique(candidate_pool["dea_source_name_raw"]),
        "dea_possible_candidate_names": collapse_unique(candidate_pool["dea_candidate_name_norm"]),
        "dea_possible_candidate_schedules": collapse_unique(candidate_pool["dea_current_schedule"]),
        "dea_possible_candidate_drug_codes": collapse_unique(candidate_pool["dea_drug_code"]),
        "dea_possible_candidate_count": candidate_pool["dea_source_name_raw"].nunique(),
        "dea_possible_match_method": "substring_candidate_unique" if candidate_pool["dea_source_name_raw"].nunique() == 1 else "substring_candidate_ambiguous",
    })

substring_candidate_tokens = pd.DataFrame(substring_candidate_rows)

display(
    substring_candidate_tokens[
        [
            "fda_ingredient_token_raw",
            "dea_possible_candidate_substance_names",
            "dea_possible_match_method",
        ]
    ].head(25)
)


,fda_ingredient_token_raw,dea_possible_candidate_substance_names,dea_possible_match_method
0,BENZHYDROCODONE HYDROCHLORIDE,Hydrocodone | Hydrocodone Bitartrate (2 1/2 H2...,substring_candidate_ambiguous
1,AMINOGLUTETHIMIDE,Glutethimide,substring_candidate_unique
2,DEXTROAMPHETAMINE ADIPATE,Amphetamine | Amphetamine Adipate | Amphetamin...,substring_candidate_ambiguous
3,DEXTROAMPHETAMINE SULFATE,Amphetamine | Amphetamine Aspartate | Amphetam...,substring_candidate_ambiguous
4,DEXTROAMPHETAMINE SACCHARATE,Amphetamine | Amphetamine Aspartate | Amphetam...,substring_candidate_ambiguous
5,DEXTROAMPHETAMINE SULFATE,Amphetamine | Amphetamine Aspartate | Amphetam...,substring_candidate_ambiguous
6,AMPHETAMINE RESIN COMPLEX,Amphetamine | Amphetamine Aspartate | Amphetam...,substring_candidate_ambiguous
7,DEXTROAMPHETAMINE RESIN COMPLEX,Amphetamine | Amphetamine Aspartate | Amphetam...,substring_candidate_ambiguous
8,AMPHETAMINE ASPARTATE/DEXTROAMPHETAMINE SULFATE,Amphetamine | Amphetamine Aspartate | Amphetam...,substring_candidate_ambiguous
9,APOMORPHINE HYDROCHLORIDE,Morphine | Morphine (H2O) | Morphine Acetate |...,substring_candidate_ambiguous



### Interpretation

This table should not be read as a set of positive controlled-substance classifications. It is a **manual-review queue**.

Some cases are plausible and substantively important, such as:

- `ARMODAFINIL` vs `MODAFINIL`
- `DEXTROAMPHETAMINE` vs `AMPHETAMINE`
- `DEXMETHYLPHENIDATE` vs `METHYLPHENIDATE`
- `BENZHYDROCODONE` vs `HYDROCODONE`

But other cases are clearly too noisy to treat as matches automatically. That is exactly why this stage is kept separate from the confident match layer.



## Aggregate the token-level results back to ingredient aggregates and then to the full FDA backbone

The FDA backbone repeats the same ingredient aggregates across many submission-event rows. So the efficient and transparent move is:

1. finish the audit at the distinct ingredient-string level
2. classify those ingredient strings
3. merge the results back to the full backbone without changing the row count

This avoids duplicating linkage work and keeps the output easy to audit.


In [11]:

EDGE_CASE_NOTES = {
    "SODIUM OXYBATE": "DEA drug-code materials distinguish GHB (Schedule I) from gamma-hydroxybutyric-acid preparations / Xyrem (Schedule III). Ingredient-only linkage identifies control-related involvement but cannot assign the final schedule confidently without preparation context.",
    "CODEINE": "DEA drug-code materials list preparation-specific schedule differences for codeine. Ingredient-only linkage can identify codeine-related control but not the final schedule without formulation / concentration context.",
    "DIHYDROCODEINE": "DEA drug-code materials list preparation-specific schedule differences for dihydrocodeine. Ingredient-only linkage can identify control-related involvement but not the final schedule without formulation / concentration context.",
    "OPIUM": "DEA drug-code materials list preparation-specific schedule differences for opium preparations. Ingredient-only linkage does not resolve that schedule variation.",
}


def collect_edge_case_notes(text: str) -> str | pd._libs.missing.NAType:
    if pd.isna(text):
        return pd.NA
    notes = []
    upper_text = str(text).upper()
    for needle, note in EDGE_CASE_NOTES.items():
        if needle in upper_text:
            notes.append(note)
    return " | ".join(dict.fromkeys(notes)) if notes else pd.NA


def aggregate_linkage_for_ingredient(group: pd.DataFrame) -> pd.Series:
    confident_tokens = group.loc[group["dea_confident_match_flag"].fillna(False)]
    list_i_tokens = group.loc[group["dea_list_i_only_flag"].fillna(False)]
    substring_tokens = group.loc[group["dea_possible_match_method"].notna()]

    return pd.Series({
        "dea_active_ingredient_token_count": group["ingredient_token_order"].size,
        "dea_confident_controlled_match_flag": not confident_tokens.empty,
        "dea_list_i_only_match_flag": confident_tokens.empty and not list_i_tokens.empty,
        "dea_possible_candidate_only_flag": confident_tokens.empty and substring_tokens.notna().any().any(),
        "dea_confident_matched_token_count": len(confident_tokens),
        "dea_list_i_token_count": len(list_i_tokens),
        "dea_possible_candidate_token_count": len(substring_tokens),
        "dea_confident_match_methods": collapse_unique(confident_tokens["dea_match_method"]),
        "dea_confident_matched_fda_tokens": collapse_unique(confident_tokens["fda_ingredient_token_raw"]),
        "dea_confident_substance_names": collapse_unique(confident_tokens["dea_scheduled_substance_names"]),
        "dea_confident_current_schedules": collapse_unique(confident_tokens["dea_scheduled_current_schedules"]),
        "dea_confident_drug_codes": collapse_unique(confident_tokens["dea_scheduled_drug_codes"]),
        "dea_list_i_substance_names": collapse_unique(list_i_tokens["dea_list_i_substance_names"]),
        "dea_list_i_drug_codes": collapse_unique(list_i_tokens["dea_list_i_drug_codes"]),
        "dea_possible_candidate_tokens": collapse_unique(substring_tokens["fda_ingredient_token_raw"]),
        "dea_possible_candidate_substance_names": collapse_unique(substring_tokens["dea_possible_candidate_substance_names"]),
        "dea_possible_candidate_methods": collapse_unique(substring_tokens["dea_possible_match_method"]),
        "dea_unmatched_fda_tokens": collapse_unique(
            group.loc[
                group["dea_confident_match_flag"].fillna(False).eq(False)
                & group["dea_list_i_only_flag"].fillna(False).eq(False)
                & group["dea_possible_match_method"].isna(),
                "fda_ingredient_token_raw",
            ]
        ),
    })


token_linkage_audit = (
    fda_ingredient_tokens
    .merge(confident_token_matches, on=TOKEN_KEY, how="left")
    .merge(
        substring_candidate_tokens[
            TOKEN_KEY + [
                "dea_possible_candidate_substance_names",
                "dea_possible_candidate_names",
                "dea_possible_candidate_schedules",
                "dea_possible_candidate_drug_codes",
                "dea_possible_candidate_count",
                "dea_possible_match_method",
            ]
        ],
        on=TOKEN_KEY,
        how="left",
    )
)

ingredient_linkage_audit = (
    token_linkage_audit
    .groupby("ActiveIngredient_list", dropna=False)
    .apply(aggregate_linkage_for_ingredient)
    .reset_index()
)

ingredient_linkage_audit["dea_edge_case_notes"] = ingredient_linkage_audit[
    "dea_confident_substance_names"
].map(collect_edge_case_notes)
ingredient_linkage_audit["dea_schedule_scope_note"] = (
    "Current DEA reference schedules only. Historical scheduling changes are not recovered in this first-pass linkage."
)
ingredient_linkage_audit["dea_linkage_status"] = np.select(
    [
        ingredient_linkage_audit["dea_confident_controlled_match_flag"],
        ingredient_linkage_audit["dea_list_i_only_match_flag"],
        ingredient_linkage_audit["dea_possible_candidate_only_flag"],
    ],
    [
        "confident_scheduled_controlled_substance_match",
        "list_i_chemical_only_match",
        "possible_parent_or_isomer_candidate_only",
    ],
    default="no_dea_signal_from_first_pass",
)

ingredient_linkage_audit["dea_uncertain_flag"] = (
    ingredient_linkage_audit["dea_possible_candidate_only_flag"]
    | ingredient_linkage_audit["dea_edge_case_notes"].notna()
)

fda_dea_linked = fda_backbone_work.merge(
    ingredient_linkage_audit,
    on="ActiveIngredient_list",
    how="left",
    validate="m:1",
)

for flag_col in [
    "dea_confident_controlled_match_flag",
    "dea_list_i_only_match_flag",
    "dea_possible_candidate_only_flag",
    "dea_uncertain_flag",
]:
    fda_dea_linked[flag_col] = fda_dea_linked[flag_col].fillna(False)

fda_dea_linked["dea_linkage_status"] = fda_dea_linked["dea_linkage_status"].fillna(
    "no_active_ingredient_available_in_fda_backbone"
)

linkage_summary = pd.DataFrame({
    "metric": [
        "FDA backbone rows",
        "rows with non-missing ActiveIngredient_list",
        "rows with missing ActiveIngredient_list",
        "rows with confident scheduled controlled-substance match",
        "rows with List I chemical-only match",
        "rows with possible substring-only candidate",
        "rows with no DEA signal from first pass",
        "rows with no active ingredient available in FDA backbone",
        "distinct ingredient aggregates with confident scheduled match",
        "distinct ingredient aggregates with List I only match",
        "distinct ingredient aggregates with possible candidate only",
    ],
    "value": [
        len(fda_dea_linked),
        int(fda_dea_linked["ActiveIngredient_list"].notna().sum()),
        int(fda_dea_linked["ActiveIngredient_list"].isna().sum()),
        int(fda_dea_linked["dea_confident_controlled_match_flag"].sum()),
        int(fda_dea_linked["dea_list_i_only_match_flag"].sum()),
        int(fda_dea_linked["dea_possible_candidate_only_flag"].sum()),
        int((fda_dea_linked["dea_linkage_status"] == "no_dea_signal_from_first_pass").sum()),
        int((fda_dea_linked["dea_linkage_status"] == "no_active_ingredient_available_in_fda_backbone").sum()),
        int(ingredient_linkage_audit["dea_confident_controlled_match_flag"].sum()),
        int(ingredient_linkage_audit["dea_list_i_only_match_flag"].sum()),
        int(ingredient_linkage_audit["dea_possible_candidate_only_flag"].sum()),
    ],
})

possible_candidate_examples = (
    fda_dea_linked.loc[fda_dea_linked["dea_possible_candidate_only_flag"]]
    .groupby(["ActiveIngredient_list", "dea_possible_candidate_substance_names"], dropna=False)
    .size()
    .reset_index(name="submission_event_rows")
    .sort_values("submission_event_rows", ascending=False)
    .head(20)
)

display(linkage_summary)
display(possible_candidate_examples)
display(ingredient_linkage_audit.head(15))


,metric,value
0,FDA backbone rows,191265
1,rows with non-missing ActiveIngredient_list,185528
2,rows with missing ActiveIngredient_list,5737
3,rows with confident scheduled controlled-subst...,17395
4,rows with List I chemical-only match,614
5,rows with possible substring-only candidate,1421
6,rows with no DEA signal from first pass,166098
7,rows with no active ingredient available in FD...,5737
8,distinct ingredient aggregates with confident ...,144
9,distinct ingredient aggregates with List I onl...,23


,ActiveIngredient_list,dea_possible_candidate_substance_names,submission_event_rows
14,DEXTROAMPHETAMINE SULFATE,Amphetamine | Amphetamine Aspartate | Amphetam...,216
11,DEXMETHYLPHENIDATE HYDROCHLORIDE,Methylphenidate | Methylphenidate Hydrochloride,154
36,METHYLTESTOSTERONE,Testosterone,129
7,BUTABARBITAL SODIUM,Barbital | Barbital Sodium,124
22,ESZOPICLONE,Zopiclone,71
46,TESTOSTERONE ENANTHATE,Testosterone,59
16,DEXTROSE; MAGNESIUM CHLORIDE; POTASSIUM CHLORI...,Amphetamine Phosphate (Monobasic),54
27,INSULIN ASPART RECOMBINANT,SPA Hydrochloride,53
26,INSULIN ASPART PROTAMINE RECOMBINANT; INSULIN ...,SPA Hydrochloride,43
34,MAGNESIUM SULFATE; POTASSIUM CHLORIDE; POTASSI...,Amphetamine Phosphate (Monobasic),37


,ActiveIngredient_list,dea_active_ingredient_token_count,dea_confident_controlled_match_flag,dea_list_i_only_match_flag,dea_possible_candidate_only_flag,dea_confident_matched_token_count,dea_list_i_token_count,dea_possible_candidate_token_count,dea_confident_match_methods,dea_confident_matched_fda_tokens,dea_confident_substance_names,dea_confident_current_schedules,dea_confident_drug_codes,dea_list_i_substance_names,dea_list_i_drug_codes,dea_possible_candidate_tokens,dea_possible_candidate_substance_names,dea_possible_candidate_methods,dea_unmatched_fda_tokens,dea_edge_case_notes,dea_schedule_scope_note,dea_linkage_status,dea_uncertain_flag
0,ABACAVIR SULFATE,1,False,False,False,0,0,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,ABACAVIR SULFATE,<NA>,Current DEA reference schedules only. Historic...,no_dea_signal_from_first_pass,False
1,ABACAVIR SULFATE; DOLUTEGRAVIR SODIUM; LAMIVUDINE,3,False,False,False,0,0,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,ABACAVIR SULFATE | DOLUTEGRAVIR SODIUM | LAMIV...,<NA>,Current DEA reference schedules only. Historic...,no_dea_signal_from_first_pass,False
2,ABACAVIR SULFATE; LAMIVUDINE,2,False,False,False,0,0,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,ABACAVIR SULFATE | LAMIVUDINE,<NA>,Current DEA reference schedules only. Historic...,no_dea_signal_from_first_pass,False
3,ABACAVIR SULFATE; LAMIVUDINE; ZIDOVUDINE,3,False,False,False,0,0,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,ABACAVIR SULFATE | LAMIVUDINE | ZIDOVUDINE,<NA>,Current DEA reference schedules only. Historic...,no_dea_signal_from_first_pass,False
4,ABACAVIR || DOLUTEGRAVIR || LAMIVUDINE,1,False,False,False,0,0,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,ABACAVIR || DOLUTEGRAVIR || LAMIVUDINE,<NA>,Current DEA reference schedules only. Historic...,no_dea_signal_from_first_pass,False
5,"ABACAVIR, DOLUTEGRAVIR, LAMIVUDINE",1,False,False,False,0,0,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,"ABACAVIR, DOLUTEGRAVIR, LAMIVUDINE",<NA>,Current DEA reference schedules only. Historic...,no_dea_signal_from_first_pass,False
6,ABACAVIR; DOLUTEGRAVIR; LAMIVUDINE,3,False,False,False,0,0,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,ABACAVIR | DOLUTEGRAVIR | LAMIVUDINE,<NA>,Current DEA reference schedules only. Historic...,no_dea_signal_from_first_pass,False
7,ABACAVIR; LAMIVUDINE,2,False,False,False,0,0,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,ABACAVIR | LAMIVUDINE,<NA>,Current DEA reference schedules only. Historic...,no_dea_signal_from_first_pass,False
8,ABACAVIR;DOLUTEGRAVIR;LAMIVUDINE,3,False,False,False,0,0,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,ABACAVIR | DOLUTEGRAVIR | LAMIVUDINE,<NA>,Current DEA reference schedules only. Historic...,no_dea_signal_from_first_pass,False
9,ABACAVIR;LAMIVUDINE,2,False,False,False,0,0,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,ABACAVIR | LAMIVUDINE,<NA>,Current DEA reference schedules only. Historic...,no_dea_signal_from_first_pass,False



### Interpretation

This is the main result of the notebook.

The linkage now distinguishes five substantively different cases:

- **confident scheduled controlled-substance match**
- **List I chemical-only match**
- **possible parent/isomer candidate only**
- **no DEA signal from the first pass**
- **no active ingredient available in the FDA backbone**

That is a much better outcome than collapsing everything into a single yes/no flag. It preserves what the current DEA-FDA bridge can support while making both unresolved DEA cases and missing-FDA-ingredient cases visible for later work.



## Export intermediate outputs

The exports below are intentionally intermediate rather than final.

### Why export now

- the DEA parsed reference should be reusable across later notebooks
- the ingredient-level audit table is the cleanest place for manual review
- the row-level linked FDA file should be available for downstream descriptive work without rewriting the backbone

### Why these exports are not placed in `clean/`

The linkage still contains uncertainty categories, schedule-scope notes, and edge cases that need later validation. So this is an **intermediate enrichment layer**, not a final cleaned analysis dataset.


In [12]:

DEA_REFERENCE_EXPORT = INTERMEDIATE_DIR / "dea_controlled_substance_reference.csv"
DEA_MANIFEST_EXPORT = INTERMEDIATE_DIR / "dea_source_manifest.csv"
FDA_DEA_TOKEN_AUDIT_EXPORT = INTERMEDIATE_DIR / "fda_dea_ingredient_token_audit.csv"
FDA_DEA_INGREDIENT_AUDIT_EXPORT = INTERMEDIATE_DIR / "fda_dea_active_ingredient_linkage_audit.csv"
FDA_DEA_LINKED_EXPORT = INTERMEDIATE_DIR / "fda_dea_controlled_substance_linkage.csv"

dea_reference.to_csv(DEA_REFERENCE_EXPORT, index=False)
dea_source_manifest.to_csv(DEA_MANIFEST_EXPORT, index=False)
token_linkage_audit.to_csv(FDA_DEA_TOKEN_AUDIT_EXPORT, index=False)
ingredient_linkage_audit.to_csv(FDA_DEA_INGREDIENT_AUDIT_EXPORT, index=False)
fda_dea_linked.to_csv(FDA_DEA_LINKED_EXPORT, index=False)

export_summary = pd.DataFrame({
    "file": [
        DEA_REFERENCE_EXPORT.name,
        DEA_MANIFEST_EXPORT.name,
        FDA_DEA_TOKEN_AUDIT_EXPORT.name,
        FDA_DEA_INGREDIENT_AUDIT_EXPORT.name,
        FDA_DEA_LINKED_EXPORT.name,
    ],
    "rows": [
        len(dea_reference),
        len(dea_source_manifest),
        len(token_linkage_audit),
        len(ingredient_linkage_audit),
        len(fda_dea_linked),
    ],
    "path": [
        str(DEA_REFERENCE_EXPORT),
        str(DEA_MANIFEST_EXPORT),
        str(FDA_DEA_TOKEN_AUDIT_EXPORT),
        str(FDA_DEA_INGREDIENT_AUDIT_EXPORT),
        str(FDA_DEA_LINKED_EXPORT),
    ],
})

display(export_summary)


,file,rows,path
0,dea_controlled_substance_reference.csv,1210,/Users/alexdelatorre/Desktop/econ580-thesis/da...
1,dea_source_manifest.csv,6,/Users/alexdelatorre/Desktop/econ580-thesis/da...
2,fda_dea_ingredient_token_audit.csv,4442,/Users/alexdelatorre/Desktop/econ580-thesis/da...
3,fda_dea_active_ingredient_linkage_audit.csv,3202,/Users/alexdelatorre/Desktop/econ580-thesis/da...
4,fda_dea_controlled_substance_linkage.csv,191265,/Users/alexdelatorre/Desktop/econ580-thesis/da...



## Final takeaway and next-step recommendations

### What we can say now

This notebook establishes a transparent first-pass DEA linkage layer for the FDA backbone. It lets us say, with different levels of confidence, which FDA submission-event records appear to involve:

- scheduled controlled substances
- DEA-regulated List I chemicals
- plausible parent/isomer/derivative candidates that still need audit

### What we still cannot say confidently

- final product-level schedule for every matched FDA record
- time-accurate historical schedule at the date of approval or submission
- full exemption treatment at the product and formulation level
- complete controlled-substance coverage when DEA general-reference lists omit certain isomers, derivatives, or preparation-specific variants
- ingredient-based linkage for FDA rows that do not carry an `ActiveIngredient_list` value in the current backbone

### Why the FDA Orange Book is not the core source here

The FDA Orange Book may later be useful for validation of approved-drug identity, marketed product configuration, or schedule-related cross-checks. But it is **not** used here to replace the FDA backbone, and it is **not** allowed to derail the core DEA linkage workflow.

### Most natural next step

The next notebook should use this intermediate linkage to produce careful descriptives of the FDA backbone split by:

- confident controlled-substance matches
- possible but uncertain controlled-substance candidates
- non-controlled or unclassified records

That is the cleanest path toward a thesis-ready descriptive analysis of whether the composition of FDA-regulated drug activity shifts after PDUFA once controlled-substance involvement is brought into the dataset.
